In [1]:
Exercice 4:

Streaming demarre


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, when, struct, to_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = (
    SparkSession.builder
    .appName("weather-transform")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

schema = StructType([
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("current", StructType([
        StructField("temperature_2m", DoubleType()),
        StructField("wind_speed_10m", DoubleType()),
        StructField("weather_code", IntegerType()),
    ])),
])

raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "weather_stream")
    .option("startingOffsets", "earliest")
    .load()
)

mesures = (
    raw.select(
        col("timestamp").alias("event_time"),
        from_json(col("value").cast("string"), schema).alias("data"),
    )
    .select(
        "event_time",
        col("data.city").alias("city"),
        col("data.country").alias("country"),
        col("data.current.temperature_2m").alias("temperature"),
        col("data.current.wind_speed_10m").alias("windspeed"),
        col("data.current.weather_code").alias("weather_code"),
    )
)

alertes = (
    mesures
    .withColumn(
        "wind_alert_level",
        when(col("windspeed") < 10, "level_0")
        .when(col("windspeed") <= 20, "level_1")
        .otherwise("level_2"),
    )
    .withColumn(
        "heat_alert_level",
        when(col("temperature") < 25, "level_0")
        .when(col("temperature") <= 35, "level_1")
        .otherwise("level_2"),
    )
)

sortie = alertes.select(to_json(struct(
    "event_time", "temperature", "windspeed", "city", "country",
    "weather_code", "wind_alert_level", "heat_alert_level",
)).alias("value"))

query = (
    sortie.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "weather_transformed")
    .option("checkpointLocation", "/home/jovyan/work/checkpoints/weather_transformed")
    .start()
)

print("Streaming demarre")

Streaming demarre


In [ ]:
Exercice 5

In [3]:
from pyspark.sql.functions import window, to_timestamp, count, avg, when
from pyspark.sql.functions import min as min_, max as max_

schema_agrege = StructType([
    StructField("event_time", StringType()),
    StructField("temperature", DoubleType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("wind_alert_level", StringType()),
    StructField("heat_alert_level", StringType()),
])

raw_agrege = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "weather_transformed")
    .option("startingOffsets", "earliest")
    .load()
)

evenements = (
    raw_agrege.select(from_json(col("value").cast("string"), schema_agrege).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp(col("event_time")))
)

agregats = (
    evenements
    .withWatermark("event_time", "1 minute")
    .groupBy(
        window(col("event_time"), "1 minute", "30 seconds").alias("fenetre"),
        col("country"),
        col("city"),
    )
    .agg(
        count(when(col("wind_alert_level") == "level_1", True)).alias("vent_level_1"),
        count(when(col("wind_alert_level") == "level_2", True)).alias("vent_level_2"),
        count(when(col("heat_alert_level") == "level_1", True)).alias("chaleur_level_1"),
        count(when(col("heat_alert_level") == "level_2", True)).alias("chaleur_level_2"),
        (
            count(when(col("wind_alert_level").isin("level_1", "level_2"), True))
            + count(when(col("heat_alert_level").isin("level_1", "level_2"), True))
        ).alias("total_alertes"),
        avg("temperature").alias("temperature_moyenne"),
        min_("temperature").alias("temperature_min"),
        max_("temperature").alias("temperature_max"),
    )
)

query_agregats = (
    agregats.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", "false")
    .option("checkpointLocation", "/home/jovyan/work/checkpoints/agregats")
    .start()
)

print("Agregats demarres")

Agregats demarres
